## PyINE trace datasets visualization

This notebook parses and displays samples/stats for a dataset of execution traces and trace deltas
generated by our proposed framework. It loads all shards of the dataset and produces both
paper-quality summary figures and detailed exploratory plots.

In [ ]:
import collections
import json

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import tqdm

import pyine.data.deltas.dataset_reader
import pyine.data.deltas.dataset_utils
import pyine.data.traces.dataset_reader
import pyine.data.traces.dataset_utils
import pyine.data.utils.splits
import pyine.utils.code.blocks
import pyine.utils.filesystem
import pyine.utils.reprod

pyine.utils.reprod.entrypoint_setup()

In [ ]:
# ------------ SETTINGS ------------
SOURCE_DATASET_NAME = "TACO"
TRACE_DATASET_PATTERN = "v1.5/10s10t.*of000026.*.lmdb"
EXPECTED_PART_COUNT = 26
SELECTED_PART_INDICES = list(range(EXPECTED_PART_COUNT))
# ----------------------------------

In [ ]:
dataset_paths = sorted(
    pyine.data.traces.dataset_utils.get_matching_dataset_paths(
        source_dataset_name=SOURCE_DATASET_NAME,
        pattern=TRACE_DATASET_PATTERN,
    )
)
assert len(dataset_paths) == EXPECTED_PART_COUNT, f"expected {EXPECTED_PART_COUNT} shards, found {len(dataset_paths)}"
dataset_paths = [dataset_paths[idx] for idx in SELECTED_PART_INDICES]

readers: list[pyine.data.traces.dataset_reader.DatasetReader] = []
for path in tqdm.tqdm(dataset_paths, desc="loading shards"):
    readers.append(pyine.data.traces.dataset_reader.DatasetReader(path))

total_traces = sum(len(r) for r in readers)
print(f"loaded {len(readers)} shards with {total_traces:,} total traces")

# display metadata from the first shard as a representative sample
metadata_fields_too_big_to_print = ["key_map", "installed_packages"]
print("\nfirst shard metadata:\n---------------------")
for metadata_key, metadata_value in readers[0].metadata.items():
    if metadata_key in metadata_fields_too_big_to_print:
        continue
    if isinstance(metadata_value, (list, dict)):
        metadata_value = json.dumps(metadata_value, indent=2)
    print(f"{metadata_key}: {metadata_value}")

In [ ]:
# extract per-trace metadata from all shards (lightweight, no full trace loading)
source_counter: collections.Counter[str] = collections.Counter()
difficulty_counter: collections.Counter[str] = collections.Counter()
traced_solution_tags: collections.Counter[str] = collections.Counter()
step_counts: list[int] = []
code_line_counts: list[int] = []
augmented_count = 0
non_augmented_count = 0
problem_ids: set[str] = set()
solution_ids: set[str] = set()
traces_per_problem: collections.Counter[str] = collections.Counter()
solutions_per_problem: dict[str, set[str]] = collections.defaultdict(set)

for reader in tqdm.tqdm(readers, desc="extracting metadata"):
    for meta in reader.trace_metadata:
        trace_id = pyine.data.traces.dataset_utils.TraceIdentifier.from_string(meta.identifier)
        problem_key = str(trace_id.get_parent_identifier().get_parent_identifier())
        solution_key = str(trace_id.get_parent_identifier())
        problem_ids.add(problem_key)
        solution_ids.add(solution_key)
        traces_per_problem[problem_key] += 1
        solutions_per_problem[problem_key].add(solution_key)
        step_counts.append(meta.step_count)
        code_line_counts.append(meta.code_string.count("\n") + 1)
        if trace_id.is_augmented:
            augmented_count += 1
        else:
            non_augmented_count += 1
        for tag in meta.tags:
            if tag.startswith("source:"):
                source_counter[tag.removeprefix("source:")] += 1
            elif tag.startswith("difficulty:"):
                difficulty_counter[tag.removeprefix("difficulty:")] += 1
            traced_solution_tags[tag] += 1

step_counts_arr = np.array(step_counts)
code_line_counts_arr = np.array(code_line_counts)
solutions_per_problem_counts = np.array([len(v) for v in solutions_per_problem.values()])
traces_per_problem_counts = np.array(list(traces_per_problem.values()))

# high-level summary
total_traces = augmented_count + non_augmented_count
print("=== Trace Dataset Summary ===")
print(f"Total traces: {total_traces:,}")
print(f"  Non-augmented: {non_augmented_count:,} ({non_augmented_count / total_traces * 100:.1f}%)")
print(f"  Augmented: {augmented_count:,} ({augmented_count / total_traces * 100:.1f}%)")
print(f"Unique problems: {len(problem_ids):,}")
print(f"Unique solutions: {len(solution_ids):,}")
print(
    f"Solutions per problem: "
    f"mean={solutions_per_problem_counts.mean():.1f}, "
    f"median={np.median(solutions_per_problem_counts):.0f}"
)
print(
    f"Traces per problem: "
    f"mean={traces_per_problem_counts.mean():.1f}, "
    f"median={np.median(traces_per_problem_counts):.0f}"
)
print(
    f"Execution steps: "
    f"mean={step_counts_arr.mean():.1f}, "
    f"median={np.median(step_counts_arr):.0f}, "
    f"max={step_counts_arr.max():,}"
)
print(f"Code lines: mean={code_line_counts_arr.mean():.1f}, median={np.median(code_line_counts_arr):.0f}")

In [ ]:
# --- paper-quality multi-panel figure ---

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

BLUE = "#4C72B0"
ORANGE = "#DD8452"
GREEN = "#55A868"
RED = "#C44E52"
PURPLE = "#8172B3"
GREY = "#8C8C8C"
YELLOW = "#CCB974"


def _sci_y(ax: plt.Axes) -> None:
    ax.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.yaxis.get_offset_text().set_fontsize(7)


fig = plt.figure(figsize=(7.2, 5.6))
gs = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.38)

# ---- (a) Difficulty distribution (pie chart) ----
ax_a = fig.add_subplot(gs[0, 0])
difficulty_order = ["EASY", "MEDIUM", "MEDIUM_HARD", "HARD", "VERY_HARD", "UNKNOWN_DIFFICULTY"]
difficulty_display = ["Easy", "Medium", "Med-Hard", "Hard", "Very Hard", "Unknown"]
diff_values = np.array([difficulty_counter.get(d, 0) for d in difficulty_order])
diff_colors = [GREEN, YELLOW, ORANGE, RED, "#8B0000", GREY]
diff_pcts = diff_values / diff_values.sum() * 100
diff_labels_with_pct = [f"{lbl}\n({pct:.1f}%)" for lbl, pct in zip(difficulty_display, diff_pcts, strict=False)]
wedges, texts = ax_a.pie(
    diff_values,
    labels=diff_labels_with_pct,
    labeldistance=1.18,
    colors=diff_colors,
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 0.8},
    textprops={"fontsize": 7.5},
)
ax_a.set_title("(a) Source problem difficulty", fontweight="bold")

# ---- (b) Execution step count distribution ----
ax_b = fig.add_subplot(gs[0, 1])
log_steps = np.log10(np.clip(step_counts_arr, 1, None))
ax_b.hist(log_steps, bins=60, color=BLUE, edgecolor="white", linewidth=0.3, alpha=0.85)
ax_b.set_xlabel("Execution steps (log$_{10}$)")
ax_b.set_ylabel("Trace count")
ax_b.set_title("(b) Execution step count", fontweight="bold")
tick_vals = [0, 1, 2, 3, 4]
ax_b.set_xticks(tick_vals)
ax_b.set_xticklabels([f"$10^{v}$" for v in tick_vals])
_sci_y(ax_b)
stats_b = f"median = {int(np.median(step_counts_arr)):,}\nmean = {int(np.mean(step_counts_arr)):,}"
ax_b.text(
    0.97,
    0.95,
    stats_b,
    transform=ax_b.transAxes,
    ha="right",
    va="top",
    fontsize=7,
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "#cccccc", "alpha": 0.9},
)

# ---- (c) Code length distribution ----
ax_c = fig.add_subplot(gs[1, 0])
sloc_clip = 100
bins_c = np.arange(0, sloc_clip + 2, 2)
ax_c.hist(
    np.clip(code_line_counts_arr, 0, sloc_clip),
    bins=bins_c,
    color=ORANGE,
    edgecolor="white",
    linewidth=0.3,
    alpha=0.85,
)
ax_c.set_xlabel("Lines of code")
ax_c.set_ylabel("Trace count")
ax_c.set_title("(c) Code length", fontweight="bold")
_sci_y(ax_c)
stats_c = f"median = {int(np.median(code_line_counts_arr))}\nmean = {code_line_counts_arr.mean():.1f}"
ax_c.text(
    0.97,
    0.95,
    stats_c,
    transform=ax_c.transAxes,
    ha="right",
    va="top",
    fontsize=7,
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "#cccccc", "alpha": 0.9},
)

# ---- (d) Traces per problem distribution (log-log) ----
ax_d = fig.add_subplot(gs[1, 1])
log_tpp = np.log10(np.clip(traces_per_problem_counts, 1, None))
ax_d.hist(log_tpp, bins=40, color=PURPLE, edgecolor="white", linewidth=0.3, alpha=0.85)
ax_d.set_xlabel("Traces per problem (log$_{10}$)")
ax_d.set_ylabel("Problem count")
ax_d.set_yscale("log")
ax_d.set_title("(d) Data density per problem", fontweight="bold")
tick_vals_d = [0, 1, 2, 3]
ax_d.set_xticks(tick_vals_d)
ax_d.set_xticklabels([f"$10^{v}$" for v in tick_vals_d])
stats_d = (
    f"median = {int(np.median(traces_per_problem_counts)):,}\n"
    f"mean = {traces_per_problem_counts.mean():.0f}\n"
    f"problems = {len(problem_ids):,}"
)
ax_d.text(
    0.97,
    0.95,
    stats_d,
    transform=ax_d.transAxes,
    ha="right",
    va="top",
    fontsize=7,
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "#cccccc", "alpha": 0.9},
)

notebook_artifacts_path = pyine.utils.filesystem.get_logs_root_path() / "paper_figures"
notebook_artifacts_path.mkdir(parents=True, exist_ok=True)
fig.savefig(notebook_artifacts_path / "dataset_stats.pdf", bbox_inches="tight")
fig.savefig(notebook_artifacts_path / "dataset_stats.png", bbox_inches="tight")
plt.show()
print(f"saved to {notebook_artifacts_path}")

In [ ]:
# --- exploratory tag plots (all tags, difficulty breakdown, source breakdown) ---

plt.figure(figsize=(12, 5))

# top-25 tags across all traces
plt.subplot(1, 2, 1)
top_tags = traced_solution_tags.most_common(25)
tag_labels = [t[0] for t in top_tags]
tag_counts = [t[1] for t in top_tags]
plt.barh(tag_labels[::-1], tag_counts[::-1])
plt.xlabel("Trace count")
plt.title("Top 25 trace tags")

# difficulty breakdown
plt.subplot(1, 2, 2)
diff_tags = {f"difficulty:{k}": v for k, v in difficulty_counter.items()}
diff_labels = list(diff_tags.keys())
diff_counts = list(diff_tags.values())
sort_idx = np.argsort(diff_counts)[::-1]
plt.bar([diff_labels[i] for i in sort_idx], [diff_counts[i] for i in sort_idx])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Count")
plt.title("Difficulty tag distribution")

plt.tight_layout()
plt.show()

# source breakdown
plt.figure(figsize=(10, 4))
src_labels = [item[0] for item in source_counter.most_common()]
src_counts = [item[1] for item in source_counter.most_common()]
plt.bar(src_labels, src_counts)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Count")
plt.title("Source tag distribution")
plt.tight_layout()
plt.show()

### Split stratification validation

Load the pre-computed split file and verify that the difficulty and solution-count
distributions are balanced across the train / valid / test subsets.

In [ ]:
split_result = pyine.data.utils.splits.get_dataset_split_result(SOURCE_DATASET_NAME)
subset_names = split_result.config.subset_names
print(f"split config: {len(split_result.identifiers)} problems across subsets {subset_names}")
print(f"stratification group rules: {split_result.config.stratif_group_rules}")

# extract per-problem difficulty and solution-count-bucket tags, grouped by subset
difficulty_tag_prefix = "difficulty:"
solution_tag_prefix = "solutions:"
subset_difficulty: dict[str, list[str]] = {name: [] for name in subset_names}
subset_solutions: dict[str, list[str]] = {name: [] for name in subset_names}
for problem_id, tags in zip(split_result.identifiers, split_result.tag_lists, strict=False):
    subset = split_result.subset_assignments.get(problem_id)
    if subset is None:
        continue
    diff_tag = next((t.removeprefix(difficulty_tag_prefix) for t in tags if t.startswith(difficulty_tag_prefix)), None)
    sol_tag = next((t.removeprefix(solution_tag_prefix) for t in tags if t.startswith(solution_tag_prefix)), None)
    if diff_tag is not None:
        subset_difficulty[subset].append(diff_tag)
    if sol_tag is not None:
        subset_solutions[subset].append(sol_tag)

# ordered category labels
difficulty_order = ["EASY", "MEDIUM", "MEDIUM_HARD", "HARD", "VERY_HARD", "UNKNOWN_DIFFICULTY"]
difficulty_display = ["Easy", "Medium", "Med-Hard", "Hard", "Very Hard", "Unknown"]
solution_bucket_order = ["0-10", "10-25", "25-50", "50-100", "100-250", "500-1000", "1000+"]

# --- figure: two side-by-side grouped bar charts ---
fig, (ax_diff, ax_sol) = plt.subplots(1, 2, figsize=(13, 4.5))
bar_width = 0.8 / len(subset_names)
all_colors = [BLUE, ORANGE, GREEN, RED, PURPLE]
subset_colors = {name: all_colors[idx % len(all_colors)] for idx, name in enumerate(subset_names)}

# (a) difficulty distribution per subset (as proportions)
for subset_idx, subset_name in enumerate(subset_names):
    counts_per_cat = collections.Counter(subset_difficulty[subset_name])
    total = max(sum(counts_per_cat.values()), 1)
    proportions = [counts_per_cat.get(cat, 0) / total for cat in difficulty_order]
    x_positions = np.arange(len(difficulty_order)) + subset_idx * bar_width
    ax_diff.bar(
        x_positions,
        proportions,
        width=bar_width,
        label=f"{subset_name} (n={total})",
        color=subset_colors[subset_name],
        alpha=0.85,
    )
ax_diff.set_xticks(np.arange(len(difficulty_order)) + bar_width * (len(subset_names) - 1) / 2)
ax_diff.set_xticklabels(difficulty_display, rotation=30, ha="right")
ax_diff.set_ylabel("Proportion of problems")
ax_diff.set_title("(a) Difficulty distribution per subset", fontweight="bold")
ax_diff.legend(fontsize=7)

# (b) solution count bucket distribution per subset (as proportions)
for subset_idx, subset_name in enumerate(subset_names):
    counts_per_cat = collections.Counter(subset_solutions[subset_name])
    total = max(sum(counts_per_cat.values()), 1)
    proportions = [counts_per_cat.get(cat, 0) / total for cat in solution_bucket_order]
    x_positions = np.arange(len(solution_bucket_order)) + subset_idx * bar_width
    ax_sol.bar(
        x_positions,
        proportions,
        width=bar_width,
        label=f"{subset_name} (n={total})",
        color=subset_colors[subset_name],
        alpha=0.85,
    )
ax_sol.set_xticks(np.arange(len(solution_bucket_order)) + bar_width * (len(subset_names) - 1) / 2)
ax_sol.set_xticklabels(solution_bucket_order, rotation=30, ha="right")
ax_sol.set_ylabel("Proportion of problems")
ax_sol.set_title("(b) Solution count distribution per subset", fontweight="bold")
ax_sol.legend(fontsize=7)

fig.suptitle("Split stratification validation", fontweight="bold", fontsize=11, y=1.01)
fig.tight_layout()
plt.show()

# summary tables as dataframes
diff_rows = {}
for subset_name in subset_names:
    counts = collections.Counter(subset_difficulty[subset_name])
    total = max(sum(counts.values()), 1)
    diff_rows[f"{subset_name} (n={total})"] = {
        display: f"{counts.get(cat, 0) / total:.3f}"
        for cat, display in zip(difficulty_order, difficulty_display, strict=False)
    }
print("Difficulty proportions per subset:")
display(pd.DataFrame(diff_rows).T)  # noqa

sol_rows = {}
for subset_name in subset_names:
    counts = collections.Counter(subset_solutions[subset_name])
    total = max(sum(counts.values()), 1)
    sol_rows[f"{subset_name} (n={total})"] = {cat: f"{counts.get(cat, 0) / total:.3f}" for cat in solution_bucket_order}
print("Solution count bucket proportions per subset:")
display(pd.DataFrame(sol_rows).T)  # noqa

In [ ]:
# load the deltas dataset (single file, separate from multi-shard traces)
try:
    deltas_path = pyine.data.deltas.dataset_utils.get_latest_dataset_path(SOURCE_DATASET_NAME)
    deltas_reader = pyine.data.deltas.dataset_reader.DatasetReader(lmdb_path=deltas_path)
    print(f"deltas dataset contains {len(deltas_reader)} traces ({deltas_path})")
    delta_generator = deltas_reader.metadata["delta_generator"]
    print(f"deltas dataset was generated using '{delta_generator}' generator")
except FileNotFoundError:
    print(f"no deltas dataset found for {SOURCE_DATASET_NAME}; skipping deltas analysis")
    deltas_reader = None

In [ ]:
# delta type and description length analysis
if deltas_reader is None:
    print("skipped; no deltas dataset loaded")
else:
    delta_type_counter: collections.Counter[str] = collections.Counter()
    delta_desc_lengths: list[int] = []
    for _trace_idx, trace_deltas in enumerate(deltas_reader):
        assert isinstance(trace_deltas, pyine.data.deltas.dataset_utils.TraceResultWithDeltas)
        for delta in trace_deltas.deltas:
            assert isinstance(delta, pyine.data.deltas.dataset_utils.TraceDelta)
            delta_type_counter[delta.event_relationship] += 1
            delta_desc_lengths.append(len(delta.variables.__repr__()))

    plt.figure(figsize=(12, 12))

    plt.subplot(2, 1, 1)
    types = list(delta_type_counter.keys())
    counts = list(delta_type_counter.values())
    plt.bar(types, counts)
    plt.xticks(rotation=45, ha="right")
    plt.xlabel("Types")
    plt.ylabel("Frequency (log-scaled)")
    plt.yscale("log")
    plt.title("Distribution of delta types")

    plt.subplot(2, 1, 2)
    plt.hist(delta_desc_lengths, bins=50)
    plt.xlabel("Description length (chars)")
    plt.ylabel("Frequency (log-scaled)")
    plt.yscale("log")
    plt.title("Distribution of delta description lengths")
    stats_text = f"Mean: {np.mean(delta_desc_lengths):.1f}\n"
    stats_text += f"Min: {np.min(delta_desc_lengths)}\n"
    stats_text += f"Max: {np.max(delta_desc_lengths)}\n"
    stats_text += f"Std: {np.std(delta_desc_lengths):.1f}"
    plt.text(
        0.95,
        0.95,
        stats_text,
        transform=plt.gca().transAxes,
        verticalalignment="top",
        horizontalalignment="right",
        bbox={"facecolor": "white", "alpha": 0.8},
    )
    plt.tight_layout()
    plt.show()